## Task 2: Design a Data Quality Audit System on Top of an ETL Pipeline

In [15]:
import requests
import pandas as pd

# Fetch 100 posts from the API
try:
    url = 'https://jsonplaceholder.typicode.com/posts'
    response = requests.get(url)
    posts = response.json()
except requests.exceptions.ConnectionError as e:
    print(f"Connection error  {e}. Retrying...")
except requests.exceptions.RequestException as e:
    print(f"An unexpected error occurred  {e}. Retrying...")
# Load into Pandas DataFrame
df = pd.DataFrame(posts)

print("Original DataFrame head:")
display(df.head())

print("\nOriginal DataFrame info:")
df.info()

Original DataFrame head:


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...



Original DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   userId  100 non-null    int64
 1   id      100 non-null    int64
 2   title   100 non-null    str  
 3   body    100 non-null    str  
dtypes: int64(2), str(2)
memory usage: 3.3 KB


In [16]:
# Initialize audit report
audit_report = {
    'before_cleaning': {
        'row_count': len(df),
        'issues': {
            'null_counts': {},
            'duplicate_rows': 0,
            'type_mismatches': {},
            'out_of_range_values': {},
            'inconsistent_string_formats': {}
        }
    },
    'after_cleaning': {
        'row_count': 0,
        'issues_fixed': {}
    }
}

print("### Data Quality Audit: Before Cleaning ###")

# 1. Null counts per column
null_counts = df.isnull().sum()
for col, count in null_counts.items():
    if count > 0:
        audit_report['before_cleaning']['issues']['null_counts'][col] = int(count)
        print(f"Column '{col}': {count} null values")

# 2. Duplicate row count
duplicate_rows = df.duplicated().sum()
audit_report['before_cleaning']['issues']['duplicate_rows'] = int(duplicate_rows)
if duplicate_rows > 0:
    print(f"Total duplicate rows: {duplicate_rows}")

# 3. Type mismatches (assuming current dtypes are expected for now, will log them)
# In a real scenario, we'd compare against a predefined schema.
# For this task, we'll just log the detected types and note if they are not as expected later.
expected_dtypes = {
    'userId': 'int64',
    'id': 'int64',
    'title': 'object',
    'body': 'object'
}

for col, dtype in df.dtypes.items():
    if str(dtype) != expected_dtypes.get(col, str(dtype)):
        audit_report['before_cleaning']['issues']['type_mismatches'][col] = f"Expected {expected_dtypes.get(col)}, found {dtype}"
    else:
        # If we just want to record the type even if it matches
        audit_report['before_cleaning']['issues']['type_mismatches'][col] = f"Type: {dtype}"

print("\nDetected data types (recorded as part of potential type mismatches):")
for col, info in audit_report['before_cleaning']['issues']['type_mismatches'].items():
    print(f"Column '{col}': {info}")

# 4. Out-of-range values (e.g., negative IDs or user IDs)
out_of_range = {}
if 'userId' in df.columns and (df['userId'] < 1).any():
    count = (df['userId'] < 1).sum()
    out_of_range['userId_negative_or_zero'] = int(count)
    print(f"'userId' contains {count} values less than 1.")
if 'id' in df.columns and (df['id'] < 1).any():
    count = (df['id'] < 1).sum()
    out_of_range['id_negative_or_zero'] = int(count)
    print(f"'id' contains {count} values less than 1.")
audit_report['before_cleaning']['issues']['out_of_range_values'] = out_of_range


# 5. Inconsistent string formats (e.g., leading/trailing spaces)
inconsistent_strings = {}
string_cols = ['title', 'body']
for col in string_cols:
    if col in df.columns and df[col].dtype == 'object':
        # Check for leading/trailing spaces
        has_leading_trailing_spaces = df[col].astype(str).apply(lambda x: x != x.strip()).sum()
        if has_leading_trailing_spaces > 0:
            inconsistent_strings[col] = f"{has_leading_trailing_spaces} values with leading/trailing spaces"
            print(f"Column '{col}': {has_leading_trailing_spaces} values with leading/trailing spaces.")

audit_report['before_cleaning']['issues']['inconsistent_string_formats'] = inconsistent_strings


print("\n--- Audit Report (Before Cleaning) --- ")
import json
print(json.dumps(audit_report['before_cleaning'], indent=4))

### Data Quality Audit: Before Cleaning ###

Detected data types (recorded as part of potential type mismatches):
Column 'userId': Type: int64
Column 'id': Type: int64
Column 'title': Expected object, found str
Column 'body': Expected object, found str

--- Audit Report (Before Cleaning) --- 
{
    "row_count": 100,
    "issues": {
        "null_counts": {},
        "duplicate_rows": 0,
        "type_mismatches": {
            "userId": "Type: int64",
            "id": "Type: int64",
            "title": "Expected object, found str",
            "body": "Expected object, found str"
        },
        "out_of_range_values": {},
        "inconsistent_string_formats": {}
    }
}


In [17]:
print("\n### Applying Transformations and Enrichments ###")

# Keep track of original df for audit
df_initial_rows = len(df)

# 1. Word count for 'body' and 'title'
df['title_word_count'] = df['title'].apply(lambda x: len(str(x).split()))
df['body_word_count'] = df['body'].apply(lambda x: len(str(x).split()))
print(f"Added 'title_word_count' and 'body_word_count' columns.")

# 2. Title casing for 'title'
# Record before and after for audit, if there are changes
original_titles = df['title'].tolist()
df['title'] = df['title'].apply(lambda x: str(x).title())
modified_titles_count = sum(1 for i, j in zip(original_titles, df['title']) if i != j)

if modified_titles_count > 0:
    audit_report['after_cleaning']['issues_fixed']['inconsistent_string_formats_title_casing'] = {
        'description': f'{modified_titles_count} titles were not in title case and were fixed.',
        'fixed_count': modified_titles_count
    }
    print(f"{modified_titles_count} titles were adjusted to title case.")

# 3. Filtering: Example - filter posts where title word count is less than 3
initial_row_count_before_filtering = len(df)
filter_condition = df['title_word_count'] >= 3
df = df[filter_condition].copy()

filtered_rows_count = initial_row_count_before_filtering - len(df)
if filtered_rows_count > 0:
    audit_report['after_cleaning']['issues_fixed']['filtered_short_titles'] = {
        'description': f'{filtered_rows_count} rows removed due to short titles (word count < 3).',
        'fixed_count': filtered_rows_count
    }
    print(f"{filtered_rows_count} rows removed due to short titles (word count < 3).")

# 4. Ranking: Rank posts by 'title_word_count' (descending)
df['post_rank'] = df['title_word_count'].rank(method='dense', ascending=False).astype(int)
print(f"Added 'post_rank' column based on 'title_word_count'.")

# Update audit report with after cleaning row count
audit_report['after_cleaning']['row_count'] = len(df)

print("\nTransformed DataFrame head:")
display(df.head())

print("\nTransformed DataFrame info:")
df.info()


### Applying Transformations and Enrichments ###
Added 'title_word_count' and 'body_word_count' columns.
100 titles were adjusted to title case.
Added 'post_rank' column based on 'title_word_count'.

Transformed DataFrame head:


,userId,id,title,body,title_word_count,body_word_count,post_rank
0,1,1,Sunt Aut Facere Repellat Provident Occaecati E...,quia et suscipit\nsuscipit recusandae consequu...,9,23,1
1,1,2,Qui Est Esse,est rerum tempore vitae\nsequi sint nihil repr...,3,31,7
2,1,3,Ea Molestias Quasi Exercitationem Repellat Qui...,et iusto sed quo iure\nvoluptatem occaecati om...,9,26,1
3,1,4,Eum Et Est Occaecati,ullam et saepe reiciendis voluptatem adipisci\...,4,28,6
4,1,5,Nesciunt Quas Odio,repudiandae veniam quaerat sunt sed\nalias aut...,3,23,7



Transformed DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   userId            100 non-null    int64
 1   id                100 non-null    int64
 2   title             100 non-null    str  
 3   body              100 non-null    str  
 4   title_word_count  100 non-null    int64
 5   body_word_count   100 non-null    int64
 6   post_rank         100 non-null    int64
dtypes: int64(5), str(2)
memory usage: 5.6 KB


In [18]:
print("\n### Generating Data Quality Audit Report ###")

# Calculate total issues before cleaning by summing explicit counts from each category
total_issues_before = 0

# Null counts
total_issues_before += sum(audit_report['before_cleaning']['issues']['null_counts'].values())

# Duplicate rows
total_issues_before += audit_report['before_cleaning']['issues']['duplicate_rows']

# Type mismatches (count entries where 'Expected' is in the value string, indicating an actual mismatch)
total_issues_before += sum(1 for k, v in audit_report['before_cleaning']['issues']['type_mismatches'].items() if 'Expected' in v)

# Out-of-range values
total_issues_before += sum(audit_report['before_cleaning']['issues']['out_of_range_values'].values())

# Inconsistent string formats (sum the counts extracted from the descriptive strings)
# The strings are like 'X values with leading/trailing spaces', so we parse X.
total_issues_before += sum(int(value.split(' ')[0]) for value in audit_report['before_cleaning']['issues']['inconsistent_string_formats'].values())

# Calculate total issues fixed
total_issues_fixed = 0
for issue_type, issue_details in audit_report['after_cleaning']['issues_fixed'].items():
    if 'fixed_count' in issue_details:
        total_issues_fixed += issue_details['fixed_count']
    else:
        total_issues_fixed += 1 # Fallback if fixed_count is not explicitly recorded

# Prepare data for the audit report table
audit_data = {
    'Metric': [
        'Initial Row Count',
        'Final Row Count',
        'Total Issues Detected (Before Cleaning)',
        'Total Issues Fixed (After Cleaning)',
        'Null Values Detected (Total)',
        'Duplicate Rows Detected',
        'Type Mismatches Detected (Total)',
        'Out-of-Range Values Detected (Total)',
        'Inconsistent String Formats Detected (Total)'
    ],
    'Value': [
        audit_report['before_cleaning']['row_count'],
        audit_report['after_cleaning']['row_count'],
        total_issues_before,
        total_issues_fixed,
        sum(audit_report['before_cleaning']['issues']['null_counts'].values()),
        audit_report['before_cleaning']['issues']['duplicate_rows'],
        sum(1 for k,v in audit_report['before_cleaning']['issues']['type_mismatches'].items() if 'Expected' in v), # Counting actual mismatches
        sum(audit_report['before_cleaning']['issues']['out_of_range_values'].values()),
        sum(int(value.split(' ')[0]) for value in audit_report['before_cleaning']['issues']['inconsistent_string_formats'].values()) # Summing the individual counts
    ]
}

# Add specific fixed issues for inconsistent string formats if applicable
if 'inconsistent_string_formats_title_casing' in audit_report['after_cleaning']['issues_fixed']:
    audit_data['Metric'].append('Titles Adjusted to Title Case')
    audit_data['Value'].append(audit_report['after_cleaning']['issues_fixed']['inconsistent_string_formats_title_casing']['fixed_count'])

# Add specific fixed issues for filtered short titles if applicable
if 'filtered_short_titles' in audit_report['after_cleaning']['issues_fixed']:
    audit_data['Metric'].append('Rows Removed Due to Short Titles')
    audit_data['Value'].append(audit_report['after_cleaning']['issues_fixed']['filtered_short_titles']['fixed_count'])


audit_df = pd.DataFrame(audit_data)

print("\n--- Data Quality Audit Report ---")
display(audit_df.to_string(index=False))

# Optionally save as CSV
# audit_df.to_csv('data_quality_audit_report.csv', index=False)
# print("Audit report saved as 'data_quality_audit_report.csv'")


### Generating Data Quality Audit Report ###

--- Data Quality Audit Report ---


'                                      Metric  Value\n                           Initial Row Count    100\n                             Final Row Count    100\n     Total Issues Detected (Before Cleaning)      2\n         Total Issues Fixed (After Cleaning)    100\n                Null Values Detected (Total)      0\n                     Duplicate Rows Detected      0\n            Type Mismatches Detected (Total)      2\n        Out-of-Range Values Detected (Total)      0\nInconsistent String Formats Detected (Total)      0\n               Titles Adjusted to Title Case    100'

In [19]:
print("\n### Loading Clean Data into MySQL ###")

# NOTE: This step requires a running MySQL instance and the appropriate credentials.
# Please ensure you have MySQL installed and a database/user configured.
# You might need to install 'mysql-connector-python'
# !pip install mysql-connector-python

import mysql.connector
import os

# MySQL connection details

try:
    # Establish connection
    conn = mysql.connector.connect(
        host='localhost',
        user='root',
        password=os.getenv("DB_PASSWORD"),
        database='data_quality_audit_db',
    )
    cursor = conn.cursor()

    # Replicate if_exists='replace' by dropping the table if it exists
    cursor.execute(f"DROP TABLE IF EXISTS posts_clean")

    # Create table with schema matching the DataFrame
    create_table_query = f"""
    CREATE TABLE posts_clean (
        userId INT,
        id INT,
        title TEXT,
        body TEXT,
        title_word_count INT,
        body_word_count INT,
        post_rank INT
    );
    """
    cursor.execute(create_table_query)

    # Prepare the INSERT statement dynamically based on DataFrame columns
    cols = ', '.join(df.columns)
    placeholders = ', '.join(['%s'] * len(df.columns))
    insert_query = f"INSERT INTO posts_clean ({cols}) VALUES ({placeholders})"

    # Convert DataFrame rows to a list of tuples for executemany
    data_to_insert = [tuple(row) for row in df.to_numpy()]

    # Execute the bulk insert
    cursor.executemany(insert_query, data_to_insert)
    conn.commit()

    print(f"Successfully loaded data into MySQL table 'posts_clean' in database 'data_quality_audit_db'.")

except mysql.connector.Error as err:
    print(f"Error connecting to MySQL or loading data: {err}")
    print("Please ensure MySQL is running, the database exists, and connection details are correct.")
    print("You might need to install 'mysql-connector-python' if you haven't already: !pip install mysql-connector-python")
finally:
    if cursor:
        cursor.close()
    if conn and conn.is_connected():
        conn.close()



### Loading Clean Data into MySQL ###
Successfully loaded data into MySQL table 'posts_clean' in database 'data_quality_audit_db'.
